# Debug Features: Correlation Analysis

**Purpose:** Diagnose if GBM features have signal for predicting PP/GLM ratio

**Steps:**
1. Sample 10% of training data
2. Check target ratio distribution
3. Calculate correlations of features with target ratio
4. Calculate correlations of features with original prediction (GLM)
5. Identify features with strong vs weak signal

In [ ]:
config_path = "config/car_liab/v1"

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import pandas as pd
import numpy as np
import yaml, sys, os
from pathlib import Path

# Find project root
cwd = Path.cwd()
for p in [cwd, cwd.parent, cwd.parent.parent]:
    if (p / 'lib').exists():
        os.chdir(p)
        break

sys.path.insert(0, str(Path.cwd() / 'lib'))
from debug_features import run_full_diagnostic

print('#' * 60)
print('# DEBUG FEATURES: CORRELATION ANALYSIS')
print('#' * 60)
print(f'Working dir: {Path.cwd()}')

In [ ]:
# Load config
pc_num = open('current.pc').read().strip()
pc_id = f'PC{pc_num}'

config_file = f'{config_path}/config.yaml'
with open(config_file, 'r') as f:
    cfg = yaml.safe_load(f)

output_base = cfg['machines'][pc_id]['paths']['output_path']
target = cfg['experiment']['target']
exposure = cfg['experiment']['exposure']
target_method = cfg['model'].get('target_method', 'direct')

print(f'\nOutput: {output_base}')
print(f'Target: {target}')
print(f'Exposure: {exposure}')
print(f'Target method: {target_method}')

In [ ]:
# Load encoded features
print(f'\n* Loading features...')
X_train = pd.read_parquet(f'{output_base}/data/04c_train_encoded.parquet')
feature_cols = X_train.columns.tolist()

print(f'  Features: {X_train.shape}')
print(f'  Feature columns: {len(feature_cols)}')

In [ ]:
# Load target and GLM
print(f'\n* Loading target and GLM...')
train_orig = pd.read_parquet(f'{output_base}/data/04b_train.parquet')
glm_train_df = pd.read_parquet(f'{output_base}/data/04c_train_glm.parquet')

# Merge on join_key to ensure proper alignment
train_aligned = train_orig[[join_key, target, exposure]].merge(glm_train_df, on=join_key, how='inner')

print(f'  Train orig: {len(train_orig)}, GLM: {len(glm_train_df)}, Aligned: {len(train_aligned)}')

# Calculate target ratio with aligned data
train_aligned['target_ratio'] = train_aligned[target] / train_aligned['glm_pred']

# Merge with features (X_train should have same rows as train from 04c)
# But to be safe, use index from train_aligned
df_combined = X_train.copy()
df_combined['target_ratio'] = train_aligned['target_ratio'].values
df_combined['glm_pred'] = train_aligned['glm_pred'].values
df_combined[exposure] = train_aligned[exposure].values

print(f'  Combined: {df_combined.shape}')
print(f'  Ratio range: {df_combined["target_ratio"].min():.3f} to {df_combined["target_ratio"].max():.3f}')


In [ ]:
# Run full diagnostic
results = run_full_diagnostic(
    df=df_combined,
    feature_cols=feature_cols,
    target_col='target_ratio',
    pred_col='glm_pred',
    weight_col=exposure,
    sample_frac=0.1,
    corr_threshold=0.05,
    chunk_size=50
)

In [ ]:
# Save results
results['corr_target'].to_csv(f'{output_base}/debug_features_corr_target.csv', index=False)
results['corr_pred'].to_csv(f'{output_base}/debug_features_corr_pred.csv', index=False)

print(f'\n* Saved:  {output_base}/debug_features_corr_*.csv')